# Framework Configuration Notebook
Central configuration for the **metadata-driven, incremental Medallion pipeline**.

## What this notebook provides
- Widget-backed parameters for storage, catalog, and environment
- Input validation (allow-list regex) to prevent SQL injection
- Helper functions reused by all framework notebooks
- All values sourced from Databricks Secrets in production

> Every framework notebook starts with `%run ./fw_0.config`

In [ ]:
# ── Widget Declarations ───────────────────────────────────────────────────────
# In production replace widget defaults with dbutils.secrets.get():
#   STORAGE_CREDENTIAL = dbutils.secrets.get(scope="finance-scope", key="storage-credential")

dbutils.widgets.text("storage_account",   "dbstoragefinance",           "Storage Account")
dbutils.widgets.text("storage_credential","dbstoragefinancestoragetoken","Storage Credential")
dbutils.widgets.text("catalog_name",       "finance_dev",                "UC Catalog")
dbutils.widgets.text("env",                "dev",                        "Environment (dev/prod)")
dbutils.widgets.text("framework_schema",   "framework",                  "Framework metadata schema")

In [ ]:
# ── Input Validation & Variable Resolution ────────────────────────────────────
import re
from datetime import datetime

def _validate_identifier(value: str, name: str) -> str:
    """Allow-list: only alphanumeric, underscore, hyphen — prevents SQL injection."""
    if not re.match(r'^[A-Za-z0-9_\-]+$', value.strip()):
        raise ValueError(
            f"Invalid widget value for '{name}': '{value}'. "
            "Only alphanumeric, underscore, and hyphen characters are permitted."
        )
    return value.strip()

STORAGE_ACCOUNT    = _validate_identifier(dbutils.widgets.get("storage_account"),    "storage_account")
STORAGE_CREDENTIAL = _validate_identifier(dbutils.widgets.get("storage_credential"), "storage_credential")
CATALOG_NAME       = _validate_identifier(dbutils.widgets.get("catalog_name"),       "catalog_name")
ENV                = _validate_identifier(dbutils.widgets.get("env"),                "env")
FRAMEWORK_SCHEMA   = _validate_identifier(dbutils.widgets.get("framework_schema"),   "framework_schema")

# Layer schema names
BRONZE_SCHEMA   = f"bronze_{ENV}"
SILVER_SCHEMA   = f"silver_{ENV}"
GOLD_SCHEMA     = f"gold_{ENV}"

# ABFS path builder
def abfs(container: str) -> str:
    return f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

BRONZE_PATH      = abfs("bronze")
SILVER_PATH      = abfs("silver")
GOLD_PATH        = abfs("gold")
CATALOG_ROOT     = abfs(f"finance-{ENV}")
CHECKPOINT_BASE  = abfs("checkpoints")
SCHEMA_BASE      = abfs("schemas")

# Fully-qualified name helper
def fq(schema: str, table: str) -> str:
    return f"{CATALOG_NAME}.{schema}.{table}"

# Framework config table FQN
FRAMEWORK_CONFIG_TABLE = fq(FRAMEWORK_SCHEMA, "pipeline_config")

print(f"Catalog          : {CATALOG_NAME}")
print(f"Framework schema : {FRAMEWORK_CONFIG_TABLE}")
print(f"Bronze path      : {BRONZE_PATH}")
print(f"Silver path      : {SILVER_PATH}")
print(f"Gold path        : {GOLD_PATH}")
print(f"Checkpoints      : {CHECKPOINT_BASE}")

In [ ]:
# ── Shared Utility Functions ──────────────────────────────────────────────────

def get_pipeline_configs(filter_active: bool = True) -> list:
    """
    Read all rows from pipeline_config.
    Returns a list of Row objects — one per source table to process.
    """
    df = spark.table(FRAMEWORK_CONFIG_TABLE)
    if filter_active:
        df = df.filter("is_active = true")
    return df.collect()


def log_pipeline_run(config_id: int, layer: str, status: str, rows_affected: int = 0, error_msg: str = None):
    """
    Append a run record to the pipeline_run_log table for observability.
    """
    from pyspark.sql import Row
    log_row = Row(
        config_id     = config_id,
        layer         = layer,
        status        = status,
        rows_affected = rows_affected,
        error_message = error_msg,
        run_timestamp = datetime.utcnow()
    )
    spark.createDataFrame([log_row]).write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(fq(FRAMEWORK_SCHEMA, "pipeline_run_log"))


def table_exists(schema: str, table: str) -> bool:
    """Check whether a Delta table exists in Unity Catalog."""
    return spark.catalog.tableExists(fq(schema, table))


print("Framework utility functions loaded.")